Import Libraries

In [ ]:
import json
from pathlib import Path 
import numpy as np
import pandas as pd
import joblib

Project Paths

In [4]:
BASE_DIR=Path.cwd()
print(BASE_DIR)
DATASET_PATH=BASE_DIR/"diabetes_binary_health_indicators_BRFSS2015.csv"
SAVE_DIR = BASE_DIR / "saved_model_v2"
SAVE_DIR.mkdir(exist_ok=True)
MODEL_PATH = SAVE_DIR / "diabetes_model.pkl"
METRICS_PATH = SAVE_DIR / "metrics.json"

E:\react\SwasthAI (1)(1)\SwasthAI\ml_service


Helper Function


In [6]:
def print_step(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

Load Dataset

In [8]:
print_step("STEP 1 : LOADING DATASET")
df = pd.read_csv(DATASET_PATH)
print(f"\nRows : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")
print("\nFirst Five Records")
print(df.head())
print("\nColumn Names")
print(df.columns.tolist())



STEP 1 : LOADING DATASET

Rows : 253680
Columns : 22

First Five Records
   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1              0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2              0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3              0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4              0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Educat

Data Validation

In [10]:
print_step("STEP 2 : DATA VALIDATION")

# Missing Values
print("\nMissing Values")
print(df.isnull().sum())

# Duplicate Rows
print("\nDuplicate Rows")
print(df.duplicated().sum())

# Data Types
print("\nData Types")
print(df.dtypes)

# Dataset Information
print("\nDataset Information")
df.info()

# Statistical Summary
print("\nStatistical Summary")
print(df.describe())

# Target Distribution
print("\nTarget Distribution")
print(df["Diabetes_binary"].value_counts())

# Target Distribution Percentage
print("\nTarget Distribution (%)")
print(
    df["Diabetes_binary"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n✓ Data Validation Completed Successfully")


STEP 2 : DATA VALIDATION

Missing Values
Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

Duplicate Rows
24206

Data Types
Diabetes_binary         float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies            

Feature Selection

In [12]:
print_step("STEP 3 : FEATURE SELECTION")

FEATURES = [
    "BMI",
    "HighBP",
    "HighChol",
    "Smoker",
    "HvyAlcoholConsump",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "GenHlth",
    "DiffWalk",
    "Age",
    "Sex"
]

TARGET = "Diabetes_binary"

# Check whether all required features are present
missing_features = []

for feature in FEATURES:

    if feature not in df.columns:

        missing_features.append(feature)

if missing_features:

    raise ValueError(
        f"Missing columns: {missing_features}"
    )

# Create input features and target variable
X = df[FEATURES]

y = df[TARGET]

print("\nSelected Features")

for feature in FEATURES:

    print(f"• {feature}")

print("\nTarget")

print(TARGET)

print("\nInput Feature Shape :", X.shape)

print("Target Shape :", y.shape)

print("\n✓ Feature Selection Completed Successfully")


STEP 3 : FEATURE SELECTION

Selected Features
• BMI
• HighBP
• HighChol
• Smoker
• HvyAlcoholConsump
• PhysActivity
• Fruits
• Veggies
• GenHlth
• DiffWalk
• Age
• Sex

Target
Diabetes_binary

Input Feature Shape : (253680, 12)
Target Shape : (253680,)

✓ Feature Selection Completed Successfully


Train-Test Split

In [14]:
from sklearn.model_selection import train_test_split
print_step("STEP 4 : TRAIN-TEST SPLIT")

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    stratify=y,

    random_state=42

)
print(f"\nTraining Samples : {len(X_train)}")

print(f"Testing Samples  : {len(X_test)}")

print(f"\nTraining Feature Shape : {X_train.shape}")

print(f"Testing Feature Shape  : {X_test.shape}")

print(f"\nTraining Target Shape : {y_train.shape}")

print(f"Testing Target Shape  : {y_test.shape}")

print("\n✓ Train-Test Split Completed Successfully")



STEP 4 : TRAIN-TEST SPLIT

Training Samples : 202944
Testing Samples  : 50736

Training Feature Shape : (202944, 12)
Testing Feature Shape  : (50736, 12)

Training Target Shape : (202944,)
Testing Target Shape  : (50736,)

✓ Train-Test Split Completed Successfully


In [15]:
print("\nTraining Target Distribution (%)")
print(
    y_train
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting Target Distribution (%)")
print(
    y_test
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Training Target Distribution (%)
Diabetes_binary
0.0    86.07
1.0    13.93
Name: proportion, dtype: float64

Testing Target Distribution (%)
Diabetes_binary
0.0    86.07
1.0    13.93
Name: proportion, dtype: float64


Feature Scaling

In [17]:
print_step("STEP 5 : FEATURE SCALING")
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\nFeature Scaling Completed Successfully")
print(f"\nScaled Training Shape : {X_train_scaled.shape}")
print(f"Scaled Testing Shape  : {X_test_scaled.shape}")
print("\n✓ Dataset Ready For Model Training")



STEP 5 : FEATURE SCALING

Feature Scaling Completed Successfully

Scaled Training Shape : (202944, 12)
Scaled Testing Shape  : (50736, 12)

✓ Dataset Ready For Model Training


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
print_step("STEP 6 : LOGISTIC REGRESSION HYPERPARAMETER TUNING")

parameter_grid = {

    "C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],

    "solver": [
        "liblinear",
        "lbfgs"
    ]

}

logistic_model = LogisticRegression(

    class_weight="balanced",

    max_iter=1000,

    random_state=42

)

logistic_search = RandomizedSearchCV(

    estimator=logistic_model,

    param_distributions=parameter_grid,

    n_iter=10,

    scoring="recall",

    cv=5,

    random_state=42,

    verbose=2,

    n_jobs=-1,

    return_train_score=True

)

print("\nSearching Best Hyperparameters...")

logistic_search.fit(

    X_train_scaled,

    y_train

)

best_logistic_model = logistic_search.best_estimator_

print("\nBest Hyperparameters")

print(logistic_search.best_params_)

print("\nBest Cross Validation Recall")

print(f"{logistic_search.best_score_:.4f}")



STEP 6 : LOGISTIC REGRESSION HYPERPARAMETER TUNING

Searching Best Hyperparameters...
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best Hyperparameters
{'solver': 'liblinear', 'C': 0.01}

Best Cross Validation Recall
0.7645


 Random Forest Hyperparameter Tuning

In [20]:
from sklearn.ensemble import RandomForestClassifier
print_step("STEP 7 : RANDOM FOREST HYPERPARAMETER TUNING")

parameter_grid = {

    "n_estimators": [
        100,
        200,
        300,
        500
    ],

    "max_depth": [
        5,
        10,
        15,
        20,  
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ],

    "max_features": [
        "sqrt",
        "log2"
    ]

}
random_forest = RandomForestClassifier(

    class_weight="balanced",

    random_state=42

)
random_forest_search = RandomizedSearchCV(

    estimator=random_forest,

    param_distributions=parameter_grid,

    n_iter=20,

    scoring="recall",

    cv=5,

    random_state=42,

    verbose=2,

    n_jobs=-1,

    return_train_score=True

)
print("\nSearching Best Hyperparameters...")

random_forest_search.fit(

    X_train,

    y_train

)
best_random_forest = random_forest_search.best_estimator_

print("\nBest Hyperparameters")

print(random_forest_search.best_params_)

print("\nBest Cross Validation Recall")

print(f"{random_forest_search.best_score_:.4f}")


STEP 7 : RANDOM FOREST HYPERPARAMETER TUNING

Searching Best Hyperparameters...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Hyperparameters
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 5}

Best Cross Validation Recall
0.7791


TEST TUNED LOGISTIC REGRESSION


In [34]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print_step("STEP 8 : TESTING TUNED LOGISTIC REGRESSION")

# Predictions
y_pred_lr = best_logistic_model.predict(X_test_scaled)

# Prediction Probabilities
y_prob_lr = best_logistic_model.predict_proba(X_test_scaled)[:, 1]

# Metrics
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
roc_auc_lr = roc_auc_score(y_test, y_prob_lr)

print(f"\nAccuracy  : {accuracy_lr:.4f}")
print(f"Precision : {precision_lr:.4f}")
print(f"Recall    : {recall_lr:.4f}")
print(f"F1 Score  : {f1_lr:.4f}")
print(f"ROC AUC   : {roc_auc_lr:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_lr))

print("\nClassification Report")
print(classification_report(y_test, y_pred_lr))


STEP 8 : TESTING TUNED LOGISTIC REGRESSION

Accuracy  : 0.7266
Precision : 0.3064
Recall    : 0.7615
F1 Score  : 0.4370
ROC AUC   : 0.8159

Confusion Matrix
[[31482 12185]
 [ 1686  5383]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.95      0.72      0.82     43667
         1.0       0.31      0.76      0.44      7069

    accuracy                           0.73     50736
   macro avg       0.63      0.74      0.63     50736
weighted avg       0.86      0.73      0.77     50736



TEST TUNED RANDOM FOREST

In [37]:
print_step("STEP 9 : TESTING TUNED RANDOM FOREST")

# Predictions
y_pred_rf = best_random_forest.predict(X_test)

# Prediction Probabilities
y_prob_rf = best_random_forest.predict_proba(X_test)[:, 1]

# Metrics
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"\nAccuracy  : {accuracy_rf:.4f}")
print(f"Precision : {precision_rf:.4f}")
print(f"Recall    : {recall_rf:.4f}")
print(f"F1 Score  : {f1_rf:.4f}")
print(f"ROC AUC   : {roc_auc_rf:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report")
print(classification_report(y_test, y_pred_rf))


STEP 9 : TESTING TUNED RANDOM FOREST

Accuracy  : 0.7131
Precision : 0.2966
Recall    : 0.7718
F1 Score  : 0.4285
ROC AUC   : 0.8123

Confusion Matrix
[[30726 12941]
 [ 1613  5456]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.95      0.70      0.81     43667
         1.0       0.30      0.77      0.43      7069

    accuracy                           0.71     50736
   macro avg       0.62      0.74      0.62     50736
weighted avg       0.86      0.71      0.76     50736



In [39]:

joblib.dump(best_random_forest, "saved_model_v2/diabetes_model.pkl")

['saved_model_v2/diabetes_model.pkl']

In [41]:
joblib.dump(scaler, "saved_model_v2/scaler.pkl")

['saved_model_v2/scaler.pkl']

In [45]:
joblib.dump(FEATURES, "saved_model_v2/feature_columns.pkl")

['saved_model_v2/feature_columns.pkl']

In [47]:
print_step("FINAL MODEL")

print("Selected Model : Tuned Random Forest")
print(f"Recall : {recall_rf:.4f}")
print(f"ROC AUC : {roc_auc_rf:.4f}")

print("\nModel Saved Successfully.")


FINAL MODEL
Selected Model : Tuned Random Forest
Recall : 0.7718
ROC AUC : 0.8123

Model Saved Successfully.
